In [12]:
# ================================
# IMPORTS
# ================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import json
import time

In [ ]:
# ================================
# CONFIGURAÇÃO DO DRIVER
# ================================

# Inicializa o navegador 
driver = webdriver.Chrome()

# Cria um "esperador" para elementos dinâmicos
wait = WebDriverWait(driver, 10)

# ================================
# 1. COLETAR TOP 250 FILMES
# ================================

# Acessa a página do Top 250
driver.get("https://www.imdb.com/chart/top/")

# Espera os elementos carregarem
filmes = wait.until(
    EC.presence_of_all_elements_located((
        By.CSS_SELECTOR,
        "li.ipc-metadata-list-summary-item"
    ))
)

print("Filmes encontrados na página:", len(filmes))


Filmes encontrados na página: 250


In [15]:
# ================================
# EXTRAIR LINKS DOS FILMES
# ================================

movie_links = []

for filme in filmes:
    try:
        # Cada item tem um <a> com o link do filme
        link = filme.find_element(By.TAG_NAME, "a").get_attribute("href")
        
        # Remove parâmetros da URL
        clean_link = link.split("?")[0]
        
        movie_links.append(clean_link)

    except:
        pass

# Remove duplicados (se houver)
movie_links = list(set(movie_links))

print("Total de links coletados:", len(movie_links))

Total de links coletados: 250


In [16]:
# ================================
# 2. FUNÇÃO DE SCRAPING DE FILME
# ================================

def scrape_movie(url):
    """
    Acessa a página de um filme e extrai:
    título, ano, nota, poster, gêneros e diretores
    """

    driver.get(url)

    # espera a página carregar
    time.sleep(2)

    # dicionário base (já trata dados faltantes)
    data = {
        "title": None,
        "year": None,
        "rating": None,
        "poster_url": None,
        "genres": [],
        "directors": [],
        "url": url
    }

    # ----------------------------
    # TÍTULO
    # ----------------------------
    try:
        title = driver.find_element(By.TAG_NAME, "h1")
        data["title"] = title.text.strip()
    except:
        pass

    # ----------------------------
    # ANO DE LANÇAMENTO
    # ----------------------------
    try:
        year = driver.find_element(By.XPATH, '//a[contains(@href,"releaseinfo")]')
        data["year"] = year.text.strip()
    except:
        pass

    # ----------------------------
    # NOTA IMDb
    # ----------------------------
    try:
        rating = driver.find_element(By.CSS_SELECTOR, '[data-testid="hero-rating-bar__aggregate-rating__score"] span')
        data["rating"] = rating.text.strip()
    except:
        pass

    # ----------------------------
    # POSTER (URL DA IMAGEM)
    # ----------------------------
    try:
        poster = driver.find_element(By.CSS_SELECTOR, 'img.ipc-image')
        data["poster_url"] = poster.get_attribute("src")
    except:
        pass

    # ----------------------------
    # GÊNEROS
    # ----------------------------
    try:
        genres = driver.find_elements(By.XPATH, '//a[contains(@href,"genres")]')
        data["genres"] = [g.text for g in genres if g.text]
    except:
        pass

    # ----------------------------
    # DIRETORES
    # ----------------------------
    try:
        # pega todos os links de pessoas
        people = driver.find_elements(By.XPATH, '//a[contains(@href,"/name/")]')
        
        directors = []

        for p in people:
            text = p.text.strip()
            
            # filtra nomes válidos (evita vazio)
            if text and text not in directors:
                directors.append(text)

        data["directors"] = directors
    except:
        pass

    return data


In [18]:
# ================================
# 3. LOOP DE SCRAPING
# ================================

movie_data = []

# Começa com 50 pra testar (depois pode usar 250)
for i, link in enumerate(movie_links[:250]):
    print(f"Processando {i+1}/50")

    data = scrape_movie(link)

    movie_data.append(data)

    # pausa para evitar bloqueio
    time.sleep(1)


# ================================
# 4. SALVAR EM JSON
# ================================

with open("movies.json", "w", encoding="utf-8") as f:
    json.dump(movie_data, f, indent=4, ensure_ascii=False)

print("Arquivo JSON gerado com sucesso!")

Processando 1/50
Processando 2/50
Processando 3/50
Processando 4/50
Processando 5/50
Processando 6/50
Processando 7/50
Processando 8/50
Processando 9/50
Processando 10/50
Processando 11/50
Processando 12/50
Processando 13/50
Processando 14/50
Processando 15/50
Processando 16/50
Processando 17/50
Processando 18/50
Processando 19/50
Processando 20/50
Processando 21/50
Processando 22/50
Processando 23/50
Processando 24/50
Processando 25/50
Processando 26/50
Processando 27/50
Processando 28/50
Processando 29/50
Processando 30/50
Processando 31/50
Processando 32/50
Processando 33/50
Processando 34/50
Processando 35/50
Processando 36/50
Processando 37/50
Processando 38/50
Processando 39/50
Processando 40/50
Processando 41/50
Processando 42/50
Processando 43/50
Processando 44/50
Processando 45/50
Processando 46/50
Processando 47/50
Processando 48/50
Processando 49/50
Processando 50/50
Processando 51/50
Processando 52/50
Processando 53/50
Processando 54/50
Processando 55/50
Processando 56/50
P

KeyboardInterrupt: 

In [19]:
driver.quit()